# 02b — Training Replicas (MLP-PLR)

Trains R MLP-PLR replicas per window pair (A, B) on **Colab GPU**, following the same
structure and output contract as notebook 02. MLP-PLR uses **Periodic → Linear → ReLU**
embeddings per numeric feature (Gorishniy et al. 2022, *On Embeddings for Numerical
Features in Tabular Deep Learning*) followed by an MLP backbone on the concatenation of
embedded numerics + raw binaries + scaled encoded categoricals.

The output schema matches notebook 02 **exactly**, so notebooks 03 and 04 consume
MLP-PLR results the same way they consume XGBoost/LR results.

## Per-replica preprocessing (Homesite specifics)

Mirrors Shoppers' MLP-PLR pattern, extended for the `X_cat` group from notebook 00:

- A fresh `OrdinalEncoder(min_frequency=1/100, handle_unknown='use_encoded_value', unknown_value=-1)` is fit on **each replica's bootstrap categorical slice**.
- Encoded categoricals are concatenated to `X` to form a `(n, n_num + n_bin + n_cat)` matrix.
- A per-replica `StandardScaler` (numerics only) is fit on the bootstrap, mirroring Shoppers.
- A second per-replica `StandardScaler` is fit on the encoded categoricals so their integer-coded magnitudes are comparable to scaled numerics.
- Binary features stay raw 0/1 (Shoppers' convention).
- Forward path: `[PLR(X_num_scaled), X_bin_raw, X_cat_scaled] → MLP backbone`. PLR is **only** applied to the true numerics; discrete categorical codes bypass it.
- Per-replica `encoder` and `cat_scaler` are bundled alongside the model artefact so explanations (nb 03/03b) and drift analysis (nb 04) can reproduce the exact training-time encoding.

## GPU reproducibility caveat

Training on GPU is not bit-reproducible even with fixed seeds — cuDNN's non-deterministic
algorithms introduce small variance that persists under `cudnn.deterministic=True`. This
low-level noise is part of the real replica-to-replica variability we're measuring.

## Seeding convention (shared with notebook 02)

For replica `r` of pair `pid`, with `SEED_BASE = 42`:
- A bootstrap seed: `SEED_BASE + pid*10_000 + r*2`
- A model seed:     `SEED_BASE + pid*10_000 + r*2 + 1`
- B bootstrap seed: `SEED_BASE + pid*10_000 + 5_000 + r*2`
- B model seed:     `SEED_BASE + pid*10_000 + 5_000 + r*2 + 1`

## Inputs / Outputs

**Input:** `data/processed/`, `data/windows/window_config.json`  
**Output per pair:** `data/models/mlp_plr/pair_{pid:02d}/`
- `replicas_A/model_r{r}.joblib`, `replicas_B/model_r{r}.joblib` — joblib bundles `{state_dict (CPU), arch_config (plain dict), scaler (StandardScaler, numerics only), cat_scaler (StandardScaler, encoded cats), encoder (OrdinalEncoder)}`
- `replicas_{A,B}/training_log_r{r}.csv` — per-epoch train loss & val PR-AUC
- `replicas_{A,B}/seeds_r{r}.json` — exact bootstrap/model seeds
- `hparams_A.json`, `hparams_B.json` — Optuna-tuned hyperparameters per window
- `reference_scaler.joblib` — `StandardScaler` fit on window A's numeric columns (parallel to nb 02; consumed by nb 04)
- `reference_encoder.joblib` — `OrdinalEncoder` fit on window A's categorical columns (parallel to nb 02; consumed by nb 04)
- `predictions.npz` — 12-key schema identical to notebook 02

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q optuna

In [ ]:
import json
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score, roc_auc_score

import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

assert torch.cuda.is_available(), 'This notebook requires a GPU runtime.'
DEVICE = torch.device('cuda')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {torch.cuda.get_device_name(0)}')

WORKSPACE = Path('/content/drive/MyDrive/Thesis/Homesite_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Fixed training knobs (not tuned) ──────────────────────────────────────
MLP_FIXED = dict(
    max_epochs        = 50,     # cap for final replica training
    tuning_max_epochs = 20,     # shorter cap during Optuna trials
    patience          = 10,     # early-stop patience on val PR-AUC
    batch_size        = 1024,
    val_tail_frac     = 0.15,   # chronological tail held out for early stopping
)

# ── Tuning configuration ─────────────────────────────────────────────────
N_TRIALS_MLP = 30   # Optuna trials per window (matches Shoppers)
CV_N_SPLITS  = 3    # TimeSeriesSplit folds (matches notebook 02)

# ── Categorical encoding configuration ─────────────────────────────────
CAT_MIN_FREQ  = 0.01   # mirrors TabReD's homesite.py recipe
CAT_UNK_VALUE = -1     # for categories absent from a bootstrap that show up at inference

print('Imports OK')

In [ ]:
# ── Model-type (hardcoded for this notebook) ───────────────────────────
MODEL_TYPE = 'mlp_plr'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODEL_TYPE : {MODEL_TYPE}')
print(f'MODEL_DIR  : {MODEL_DIR}')

In [ ]:
# ── Load processed data ────────────────────────────────────────────────
X_df = pd.read_parquet(PROC_DIR / 'X.parquet')
X    = X_df.values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

# Categoricals are stored as raw strings (object dtype) in X_cat.parquet.
X_cat_df = pd.read_parquet(PROC_DIR / 'X_cat.parquet')
X_cat    = np.asarray(X_cat_df, dtype=object)

with open(PROC_DIR / 'feature_names.json') as f:
    feat_names = json.load(f)

# Column positions in the combined model input matrix [X | X_cat_encoded].
# X.parquet's columns are num + bin (in that order, per notebook 00). Encoded cats
# are stacked at the end, so cat positions in the combined matrix start at X.shape[1].
num_col_idx          = [X_df.columns.get_loc(c) for c in feat_names['num']]
bin_col_idx          = [i for i in range(X.shape[1]) if i not in set(num_col_idx)]
n_num                = len(num_col_idx)
n_bin                = len(bin_col_idx)
n_cat                = X_cat.shape[1]
n_num_bin            = X.shape[1]                           # columns in X.parquet
cat_col_idx_combined = list(range(n_num_bin, n_num_bin + n_cat))   # positions in combined matrix
n_total              = n_num_bin + n_cat

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)
R       = config['parameters']['R']
K_FRAC  = config['parameters']['K_FRAC']
pairs   = config['pairs']

print(f'X     : {X.shape}    ({n_num} num + {n_bin} bin)')
print(f'X_cat : {X_cat.shape}  ({n_cat} cat features as raw strings)')
print(f'Y     : {Y.shape}')
print(f'R={R}, K_FRAC={K_FRAC}, {len(pairs)} window pairs')
print(f'Combined feature count fed to model: {n_total}')

assert n_num + n_bin == X.shape[1], 'num/bin partition does not cover all of X.'
assert len(set(num_col_idx) & set(bin_col_idx)) == 0, 'num/bin column indices overlap.'
assert X.shape[0] == Y.shape[0] == X_cat.shape[0], 'X / Y / X_cat row counts inconsistent.'
assert X.shape[1] == len(feat_names['all']),       'X.parquet does not match feature_names["all"].'
assert list(X_df.columns)     == feat_names['all'], 'X.parquet column order mismatch.'
assert list(X_cat_df.columns) == feat_names['cat'], 'X_cat.parquet column order mismatch.'
assert len(pairs) > 0, 'No rolling-window pairs found.'

## Model definitions

Note: keep these classes byte-identical with notebook 03's copy (used during SHAP/explanation extraction).

In [ ]:
class PLREmbedding(nn.Module):
    """Vectorised PLR embedding for all numeric features at once.

    Per feature (no weight sharing across features):
        periodic(x)  :  v = 2π · c · x        c ∈ ℝ^k, c ~ N(0, sigma_init^2)
                        [sin(v), cos(v)]      → ℝ^(2k)
        linear       :  W·[sin,cos] + b       W ∈ ℝ^(2k × d), b ∈ ℝ^d
        relu         :  max(0, ·)             → ℝ^d
    """

    def __init__(self, n_num_features: int, k_periodic: int,
                 sigma_init: float, d_embedding: int):
        super().__init__()
        self.n_num_features = n_num_features
        self.k_periodic     = k_periodic
        self.d_embedding    = d_embedding

        # Per-feature coefficient vectors c ~ N(0, sigma_init^2)
        self.c = nn.Parameter(torch.randn(n_num_features, k_periodic) * sigma_init)

        # Per-feature linear weights & biases (no weight sharing)
        self.W = nn.Parameter(torch.empty(n_num_features, 2 * k_periodic, d_embedding))
        self.b = nn.Parameter(torch.zeros(n_num_features, d_embedding))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x_num: torch.Tensor) -> torch.Tensor:
        # x_num: (batch, n_num_features)
        v  = (2.0 * math.pi) * self.c.unsqueeze(0) * x_num.unsqueeze(-1)
        pe = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)
        out = torch.einsum('bnk,nkd->bnd', pe, self.W) + self.b.unsqueeze(0)
        out = torch.relu(out)
        return out.reshape(out.shape[0], -1)


class MLPPLR(nn.Module):
    """MLP-PLR with three input groups: numerics (PLR-embedded), binaries (raw 0/1),
    and encoded categoricals (scaled, bypass PLR).

    Outputs a single logit per instance (no sigmoid — use BCEWithLogitsLoss).
    Column routing is stored as non-trainable buffers so it travels with state_dict.
    """

    def __init__(self, n_num_features: int, n_bin_features: int, n_cat_features: int,
                 k_periodic: int, sigma_init: float, d_embedding: int,
                 n_layers: int, hidden_dim: int, dropout: float,
                 num_col_idx, bin_col_idx, cat_col_idx):
        super().__init__()
        self.n_num_features = n_num_features
        self.n_bin_features = n_bin_features
        self.n_cat_features = n_cat_features

        # Column routing as buffers (survive state_dict save/load).
        self.register_buffer('num_col_idx_buf', torch.as_tensor(num_col_idx, dtype=torch.long))
        self.register_buffer('bin_col_idx_buf', torch.as_tensor(bin_col_idx, dtype=torch.long))
        self.register_buffer('cat_col_idx_buf', torch.as_tensor(cat_col_idx, dtype=torch.long))

        self.plr = PLREmbedding(n_num_features, k_periodic, sigma_init, d_embedding)

        # MLP backbone — input combines: PLR-embedded numerics + raw binaries + scaled cats.
        input_dim = n_num_features * d_embedding + n_bin_features + n_cat_features
        layers = []
        in_dim = input_dim
        for _ in range(n_layers):
            layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.head     = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, n_num + n_bin + n_cat) — combined matrix [X_scaled | X_cat_scaled].
        x_num = x.index_select(1, self.num_col_idx_buf)   # (batch, n_num)
        x_bin = x.index_select(1, self.bin_col_idx_buf)   # (batch, n_bin) — raw 0/1
        x_cat = x.index_select(1, self.cat_col_idx_buf)   # (batch, n_cat) — scaled
        emb   = self.plr(x_num)                            # (batch, n_num * d_emb)
        z     = torch.cat([emb, x_bin, x_cat], dim=-1)     # (batch, input_dim)
        h     = self.backbone(z)                           # (batch, hidden_dim)
        return self.head(h).squeeze(-1)                    # (batch,)


print('Model classes defined.')

## Helper functions

`stratified_bootstrap` and `compute_flagged_topk` are byte-identical with notebook 02 so cross-model-type replica alignment holds.

In [ ]:
def stratified_bootstrap(idx: np.ndarray, Y: np.ndarray, seed: int) -> np.ndarray:
    """Stratified bootstrap: sample with replacement, preserving class ratio."""
    rng = np.random.default_rng(seed)
    pos = idx[Y[idx] == 1]
    neg = idx[Y[idx] == 0]
    boot_pos = rng.choice(pos, size=len(pos), replace=True)
    boot_neg = rng.choice(neg, size=len(neg), replace=True)
    out = np.concatenate([boot_pos, boot_neg])
    rng.shuffle(out)
    return out


def compute_flagged_topk(p_hat_A, p_hat_B, k_frac):
    """Top-K% flagged set ranked by max(p_hat_A, p_hat_B)."""
    assert p_hat_A.shape == p_hat_B.shape
    n = len(p_hat_A)
    k = max(int(np.ceil(k_frac * n)), 1)
    score = np.maximum(p_hat_A, p_hat_B)
    topk_unsorted = np.argpartition(-score, k - 1)[:k]
    return np.sort(topk_unsorted)


def make_encoder() -> OrdinalEncoder:
    """Fresh OrdinalEncoder configured with the TabReD-style frequency threshold."""
    return OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=CAT_UNK_VALUE,
        min_frequency=CAT_MIN_FREQ,
        encoded_missing_value=CAT_UNK_VALUE,
    )


def assemble_combined(X_num_part: np.ndarray,
                      X_cat_str_part: np.ndarray,
                      encoder: OrdinalEncoder,
                      fit: bool) -> np.ndarray:
    """Encode the categorical part and horizontally stack with X (numerics + binaries).

    Returns an unscaled `(n, n_num + n_bin + n_cat)` float32 matrix in the order that
    `num_col_idx`, `bin_col_idx`, `cat_col_idx_combined` index into.
    """
    if fit:
        X_cat_enc = encoder.fit_transform(X_cat_str_part)
    else:
        X_cat_enc = encoder.transform(X_cat_str_part)
    return np.hstack([X_num_part, X_cat_enc.astype(np.float32)])


def scale_combined_inplace(X_combined: np.ndarray,
                           num_scaler: StandardScaler,
                           cat_scaler: StandardScaler,
                           num_idx,
                           cat_idx) -> np.ndarray:
    """Scale numeric and categorical columns; binary columns pass through untouched.

    Mirrors Shoppers' `scale_numeric_inplace` but extended for the cat group.
    """
    out = X_combined.copy()
    out[:, num_idx] = num_scaler.transform(X_combined[:, num_idx])
    out[:, cat_idx] = cat_scaler.transform(X_combined[:, cat_idx])
    return out


def build_arch_config(hparams: dict) -> dict:
    """Assemble the dict that reconstructs an MLPPLR from arch-only hyperparameters."""
    return {
        'n_num_features': n_num,
        'n_bin_features': n_bin,
        'n_cat_features': n_cat,
        'k_periodic':     int(hparams['k_periodic']),
        'sigma_init':     float(hparams['sigma_init']),
        'd_embedding':    int(hparams['d_embedding']),
        'n_layers':       int(hparams['n_layers']),
        'hidden_dim':     int(hparams['hidden_dim']),
        'dropout':        float(hparams['dropout']),
        'num_col_idx':    [int(i) for i in num_col_idx],
        'bin_col_idx':    [int(i) for i in bin_col_idx],
        'cat_col_idx':    [int(i) for i in cat_col_idx_combined],
    }


def train_one_mlp_plr(X_tr_tensor: torch.Tensor,
                      Y_tr_tensor: torch.Tensor,
                      arch_config: dict,
                      train_config: dict,
                      model_seed: int,
                      X_va_tensor: torch.Tensor = None,
                      Y_va_tensor: torch.Tensor = None):
    """Train one MLP-PLR replica with patience-based early stopping.

    When X_va_tensor / Y_va_tensor are provided (final replica training), those tensors are
    used directly as the early-stopping validation set — they must be the chronological
    tail of the training window, scaled with the per-replica scalers, and extracted
    BEFORE bootstrapping (see main loop).

    When X_va_tensor is None (Optuna tuning phase), the function falls back to an internal
    tail split: the last `val_tail_frac` fraction of X_tr_tensor is held out.

    Returns: (best_state_dict_on_cpu, training_log_df, best_epoch, best_val_pr_auc)
    """
    torch.manual_seed(model_seed)
    torch.cuda.manual_seed_all(model_seed)
    np.random.seed(model_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    if X_va_tensor is None:
        n     = X_tr_tensor.shape[0]
        n_val = max(int(np.ceil(n * train_config['val_tail_frac'])), 1)
        X_tr  = X_tr_tensor[:-n_val]
        Y_tr  = Y_tr_tensor[:-n_val]
        X_va  = X_tr_tensor[-n_val:]
        Y_va  = Y_tr_tensor[-n_val:]
    else:
        X_tr, Y_tr = X_tr_tensor, Y_tr_tensor
        X_va, Y_va = X_va_tensor, Y_va_tensor

    model = MLPPLR(**arch_config).to(DEVICE)
    opt   = torch.optim.AdamW(
        model.parameters(),
        lr=train_config['learning_rate'],
        weight_decay=train_config['weight_decay'],
    )
    loss_fn = nn.BCEWithLogitsLoss()

    best_val_pr   = -1.0
    best_state    = None
    best_epoch    = -1
    patience_left = train_config['patience']
    bs            = train_config['batch_size']
    n_tr          = X_tr.shape[0]

    log_rows = []

    for epoch in range(1, train_config['max_epochs'] + 1):
        model.train()
        perm = torch.randperm(n_tr, device=DEVICE)
        total_loss = 0.0
        for start in range(0, n_tr, bs):
            sel   = perm[start:start + bs]
            xb    = X_tr[sel]
            yb    = Y_tr[sel]
            opt.zero_grad()
            logit = model(xb)
            loss  = loss_fn(logit, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * xb.shape[0]
        avg_loss = total_loss / n_tr

        model.eval()
        val_probs_chunks = []
        with torch.no_grad():
            for start in range(0, X_va.shape[0], 2048):
                logit = model(X_va[start:start + 2048])
                val_probs_chunks.append(torch.sigmoid(logit).cpu().numpy())
        val_probs = np.concatenate(val_probs_chunks)
        val_y     = Y_va.cpu().numpy()
        val_pr    = float(average_precision_score(val_y, val_probs))

        log_rows.append({'epoch': epoch, 'train_loss': avg_loss, 'val_pr_auc': val_pr})

        if val_pr > best_val_pr:
            best_val_pr   = val_pr
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch    = epoch
            patience_left = train_config['patience']
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = len(log_rows)

    del model
    torch.cuda.empty_cache()

    return best_state, pd.DataFrame(log_rows), best_epoch, best_val_pr


def predict_proba_mlp_plr(model: nn.Module, X_tensor: torch.Tensor,
                          batch_size: int = 2048) -> np.ndarray:
    """Forward pass in eval mode, apply sigmoid, return 1-D numpy float32."""
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, X_tensor.shape[0], batch_size):
            logit = model(X_tensor[start:start + batch_size])
            out.append(torch.sigmoid(logit).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def tune_hyperparameters_mlp_plr(window_idx: np.ndarray,
                                  X_all: np.ndarray,
                                  X_cat_all: np.ndarray,
                                  Y_all: np.ndarray,
                                  study_seed: int,
                                  n_trials: int = N_TRIALS_MLP) -> dict:
    """Optuna MLP-PLR tuning on the given training window.

    Per CV fold: a fresh encoder + numeric scaler + cat scaler are fit on the fold's
    training portion only, so all preprocessing stays leak-free across folds.
    """
    X_win     = X_all[window_idx]      # preserves chronological step-sorted order
    X_cat_win = X_cat_all[window_idx]
    Y_win     = Y_all[window_idx]

    tscv = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    def objective(trial: optuna.Trial) -> float:
        hp = {
            'k_periodic':    trial.suggest_int('k_periodic', 4, 16),
            'sigma_init':    trial.suggest_float('sigma_init', 0.01, 10.0, log=True),
            'd_embedding':   trial.suggest_categorical('d_embedding', [8, 16, 24]),
            'n_layers':      trial.suggest_int('n_layers', 1, 3),
            'hidden_dim':    trial.suggest_categorical('hidden_dim', [128, 256, 512]),
            'dropout':       trial.suggest_float('dropout', 0.0, 0.5),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
            'weight_decay':  trial.suggest_float('weight_decay',  1e-6, 1e-2, log=True),
        }

        arch_config  = build_arch_config(hp)
        train_config = {
            'learning_rate': hp['learning_rate'],
            'weight_decay':  hp['weight_decay'],
            'max_epochs':    MLP_FIXED['tuning_max_epochs'],
            'patience':      MLP_FIXED['patience'],
            'batch_size':    MLP_FIXED['batch_size'],
            'val_tail_frac': MLP_FIXED['val_tail_frac'],
        }

        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            # Per-fold preprocessing fit on fold-train only.
            encoder = make_encoder()
            X_tr_combined_unscaled = assemble_combined(X_win[tr], X_cat_win[tr], encoder, fit=True)
            X_va_combined_unscaled = assemble_combined(X_win[va], X_cat_win[va], encoder, fit=False)

            num_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, num_col_idx])
            cat_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, cat_col_idx_combined])

            X_tr_s = scale_combined_inplace(X_tr_combined_unscaled, num_scaler, cat_scaler,
                                             num_col_idx, cat_col_idx_combined)
            X_va_s = scale_combined_inplace(X_va_combined_unscaled, num_scaler, cat_scaler,
                                             num_col_idx, cat_col_idx_combined)

            X_tr_t = torch.from_numpy(X_tr_s).float().to(DEVICE)
            Y_tr_t = torch.from_numpy(Y_win[tr].astype(np.float32)).to(DEVICE)
            X_va_t = torch.from_numpy(X_va_s).float().to(DEVICE)

            best_state, _, _, _ = train_one_mlp_plr(
                X_tr_t, Y_tr_t, arch_config, train_config,
                model_seed=study_seed + fold,
            )

            pred_model = MLPPLR(**arch_config).to(DEVICE)
            pred_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
            probs = predict_proba_mlp_plr(pred_model, X_va_t)
            pr_aucs.append(float(average_precision_score(Y_win[va], probs)))

            del pred_model, X_tr_t, Y_tr_t, X_va_t
            torch.cuda.empty_cache()

        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study   = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)


def assert_binary_slice(y, name):
    assert len(np.unique(y)) == 2, f'{name} contains only one class.'


print('Helpers defined.')

## Main training loop — one iteration per window pair

In [ ]:
SEED_BASE = 42   # Shared with notebook 02 — do not change.

CURRENT_RUN_PARAMS = {
    'model_type':  MODEL_TYPE,
    'seed_base':   SEED_BASE,

    'window_parameters': {
        'L':           int(config['parameters']['L']),
        'S':           int(config['parameters']['S']),
        'H':           int(config['parameters']['H']),
        'R':           int(R),
        'K_FRAC':      float(K_FRAC),
        'PAIR_STRIDE': int(config['parameters'].get('PAIR_STRIDE', 1)),
    },

    'cv_n_splits':   int(CV_N_SPLITS),
    'n_trials_mlp':  int(N_TRIALS_MLP),
    'mlp_fixed':     MLP_FIXED,
    'cat_min_freq':  float(CAT_MIN_FREQ),
    'cat_unk_value': int(CAT_UNK_VALUE),
}

performance_log = []

for p in pairs:
    pid             = p['pair_id']
    pair_dir        = MODEL_DIR / f'pair_{pid:02d}'
    pred_file       = pair_dir / 'predictions.npz'
    run_params_file = pair_dir / 'run_params.json'

    # ── Skip-if-done: validate full run configuration against saved run_params ──
    if pred_file.exists() and run_params_file.exists():
        with open(run_params_file) as f:
            saved = json.load(f)
        params_match = (saved == CURRENT_RUN_PARAMS)

        if params_match:
            data = np.load(pred_file)
            if all(k in data.files for k in ('roc_auc_A', 'roc_auc_B')):
                print(f'Pair {pid:02d}: already done, skipping.')
                performance_log.append({
                    'pair_id':   pid,
                    'pr_auc_A':  float(data['pr_auc_A']),
                    'pr_auc_B':  float(data['pr_auc_B']),
                    'roc_auc_A': float(data['roc_auc_A']),
                    'roc_auc_B': float(data['roc_auc_B']),
                    'n_flagged': int(data['flagged_idx'].shape[0]),
                })
                continue
            else:
                print(f'Pair {pid:02d}: stale predictions.npz (missing roc_auc keys) — re-running.')
        else:
            changed_keys = [k for k in CURRENT_RUN_PARAMS if saved.get(k) != CURRENT_RUN_PARAMS[k]]
            print(f'Pair {pid:02d}: run_params changed in {changed_keys} — re-running.')
    elif pred_file.exists():
        print(f'Pair {pid:02d}: run_params.json missing — re-running.')

    print(f'\n── [mlp_plr] Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]}  '
          f'eval={p["eval_start_label"]}→{p["eval_end_label"]}  '
          f'|A|={p["n_train_A"]:,} |B|={p["n_train_B"]:,} |eval|={p["n_eval"]:,} ──')

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    assert len(set(idx_A.tolist()) & set(idx_eval.tolist())) == 0, 'idx_A overlaps idx_eval!'
    assert len(set(idx_B.tolist()) & set(idx_eval.tolist())) == 0, 'idx_B overlaps idx_eval!'

    X_eval     = X[idx_eval]
    X_cat_eval = X_cat[idx_eval]
    Y_eval     = Y[idx_eval]
    assert_binary_slice(Y[idx_A], f'pair {pid:02d} window A')
    assert_binary_slice(Y[idx_B], f'pair {pid:02d} window B')
    assert_binary_slice(Y_eval,   f'pair {pid:02d} eval slice')

    pair_dir.mkdir(parents=True, exist_ok=True)
    dir_A = pair_dir / 'replicas_A'; dir_A.mkdir(exist_ok=True)
    dir_B = pair_dir / 'replicas_B'; dir_B.mkdir(exist_ok=True)

    # ── Reference scaler (numerics) and reference encoder (cats) on window A ──
    # Both consumed by nb 04 for cross-pair drift; parallel to nb 02.
    ref_scaler = StandardScaler().fit(X[idx_A][:, num_col_idx])
    joblib.dump(ref_scaler, pair_dir / 'reference_scaler.joblib')

    ref_encoder = make_encoder()
    ref_encoder.fit(X_cat[idx_A])
    joblib.dump(ref_encoder, pair_dir / 'reference_encoder.joblib')

    # ── Tune hyperparameters per window ────────────────────────────────────
    print('  Tuning window A ...')
    hparams_A = tune_hyperparameters_mlp_plr(
        idx_A, X, X_cat, Y, study_seed=SEED_BASE + pid * 10 + 1, n_trials=N_TRIALS_MLP)
    with open(pair_dir / 'hparams_A.json', 'w') as f:
        json.dump(hparams_A, f, indent=2)
    print(f'    best A params: {hparams_A}')

    print('  Tuning window B ...')
    hparams_B = tune_hyperparameters_mlp_plr(
        idx_B, X, X_cat, Y, study_seed=SEED_BASE + pid * 10 + 2, n_trials=N_TRIALS_MLP)
    with open(pair_dir / 'hparams_B.json', 'w') as f:
        json.dump(hparams_B, f, indent=2)
    print(f'    best B params: {hparams_B}')

    # ── Chronological ES validation split (done before bootstrap, shared across replicas) ──
    val_frac = MLP_FIXED['val_tail_frac']

    n_es_A           = max(int(np.ceil(len(idx_A) * val_frac)), 1)
    idx_A_boot       = idx_A[:-n_es_A]                # bootstrap source (excludes chronological tail)
    X_va_A_raw       = X[idx_A[-n_es_A:]]             # raw ES validation features (scaled per-replica)
    X_cat_va_A_raw   = X_cat[idx_A[-n_es_A:]]
    Y_va_A           = Y[idx_A[-n_es_A:]].astype(np.float32)

    n_es_B           = max(int(np.ceil(len(idx_B) * val_frac)), 1)
    idx_B_boot       = idx_B[:-n_es_B]
    X_va_B_raw       = X[idx_B[-n_es_B:]]
    X_cat_va_B_raw   = X_cat[idx_B[-n_es_B:]]
    Y_va_B           = Y[idx_B[-n_es_B:]].astype(np.float32)

    assert_binary_slice(Y_va_A,        f'pair {pid:02d} window A early-stopping tail')
    assert_binary_slice(Y_va_B,        f'pair {pid:02d} window B early-stopping tail')
    assert_binary_slice(Y[idx_A_boot], f'pair {pid:02d} window A bootstrap source')
    assert_binary_slice(Y[idx_B_boot], f'pair {pid:02d} window B bootstrap source')

    # ── Train R replicas for window A ──────────────────────────────────────
    preds_A          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_A = np.zeros(R, dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + r * 2          # A: even offset
        model_seed = SEED_BASE + pid * 10_000 + r * 2 + 1      # A: odd offset

        boot_idx = stratified_bootstrap(idx_A_boot, Y, seed=boot_seed)

        # Per-replica encoder + scalers, fit on the bootstrap training portion only.
        encoder    = make_encoder()
        X_tr_combined_unscaled = assemble_combined(X[boot_idx],   X_cat[boot_idx],   encoder, fit=True)
        X_va_combined_unscaled = assemble_combined(X_va_A_raw,    X_cat_va_A_raw,    encoder, fit=False)
        X_eval_combined_unscaled = assemble_combined(X_eval,       X_cat_eval,        encoder, fit=False)

        num_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, num_col_idx])
        cat_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, cat_col_idx_combined])

        X_tr_s   = scale_combined_inplace(X_tr_combined_unscaled,   num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)
        X_va_s   = scale_combined_inplace(X_va_combined_unscaled,   num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)
        X_eval_s = scale_combined_inplace(X_eval_combined_unscaled, num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)

        X_tr_t   = torch.from_numpy(X_tr_s).float().to(DEVICE)
        Y_tr_t   = torch.from_numpy(Y[boot_idx].astype(np.float32)).to(DEVICE)
        X_va_t   = torch.from_numpy(X_va_s).float().to(DEVICE)
        Y_va_t   = torch.from_numpy(Y_va_A).to(DEVICE)
        X_eval_t = torch.from_numpy(X_eval_s).float().to(DEVICE)

        arch_config  = build_arch_config(hparams_A)
        train_config = {
            'learning_rate': hparams_A['learning_rate'],
            'weight_decay':  hparams_A['weight_decay'],
            'max_epochs':    MLP_FIXED['max_epochs'],
            'patience':      MLP_FIXED['patience'],
            'batch_size':    MLP_FIXED['batch_size'],
            'val_tail_frac': val_frac,
        }

        best_state, training_log, best_epoch, best_val_pr = train_one_mlp_plr(
            X_tr_t, Y_tr_t, arch_config, train_config, model_seed=model_seed,
            X_va_tensor=X_va_t, Y_va_tensor=Y_va_t,
        )
        training_log.to_csv(dir_A / f'training_log_r{r}.csv', index=False)

        # Reconstruct best model for prediction.
        pred_model = MLPPLR(**arch_config).to(DEVICE)
        pred_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
        preds_A[r] = predict_proba_mlp_plr(pred_model, X_eval_t)
        per_rep_pr_auc_A[r] = float(average_precision_score(Y_eval, preds_A[r]))

        # Bundle per-replica artefacts. 'scaler' key stays numerics-only (Shoppers parity);
        # cat_scaler and encoder are extra keys for the cat path.
        bundle = {
            'state_dict':  best_state,
            'arch_config': arch_config,
            'scaler':      num_scaler,
            'cat_scaler':  cat_scaler,
            'encoder':     encoder,
        }
        joblib.dump(bundle, dir_A / f'model_r{r}.joblib', compress=3)

        with open(dir_A / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)

        print(f'  A replica {r}: PR-AUC = {per_rep_pr_auc_A[r]:.4f}  '
              f'(best epoch {best_epoch}, val PR-AUC {best_val_pr:.4f})')

        del pred_model, X_tr_t, Y_tr_t, X_va_t, Y_va_t, X_eval_t
        torch.cuda.empty_cache()

    # ── Train R replicas for window B ──────────────────────────────────────
    preds_B          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_B = np.zeros(R, dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + 5_000 + r * 2
        model_seed = SEED_BASE + pid * 10_000 + 5_000 + r * 2 + 1

        boot_idx = stratified_bootstrap(idx_B_boot, Y, seed=boot_seed)

        encoder    = make_encoder()
        X_tr_combined_unscaled = assemble_combined(X[boot_idx],   X_cat[boot_idx],   encoder, fit=True)
        X_va_combined_unscaled = assemble_combined(X_va_B_raw,    X_cat_va_B_raw,    encoder, fit=False)
        X_eval_combined_unscaled = assemble_combined(X_eval,       X_cat_eval,        encoder, fit=False)

        num_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, num_col_idx])
        cat_scaler = StandardScaler().fit(X_tr_combined_unscaled[:, cat_col_idx_combined])

        X_tr_s   = scale_combined_inplace(X_tr_combined_unscaled,   num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)
        X_va_s   = scale_combined_inplace(X_va_combined_unscaled,   num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)
        X_eval_s = scale_combined_inplace(X_eval_combined_unscaled, num_scaler, cat_scaler, num_col_idx, cat_col_idx_combined)

        X_tr_t   = torch.from_numpy(X_tr_s).float().to(DEVICE)
        Y_tr_t   = torch.from_numpy(Y[boot_idx].astype(np.float32)).to(DEVICE)
        X_va_t   = torch.from_numpy(X_va_s).float().to(DEVICE)
        Y_va_t   = torch.from_numpy(Y_va_B).to(DEVICE)
        X_eval_t = torch.from_numpy(X_eval_s).float().to(DEVICE)

        arch_config  = build_arch_config(hparams_B)
        train_config = {
            'learning_rate': hparams_B['learning_rate'],
            'weight_decay':  hparams_B['weight_decay'],
            'max_epochs':    MLP_FIXED['max_epochs'],
            'patience':      MLP_FIXED['patience'],
            'batch_size':    MLP_FIXED['batch_size'],
            'val_tail_frac': val_frac,
        }

        best_state, training_log, best_epoch, best_val_pr = train_one_mlp_plr(
            X_tr_t, Y_tr_t, arch_config, train_config, model_seed=model_seed,
            X_va_tensor=X_va_t, Y_va_tensor=Y_va_t,
        )
        training_log.to_csv(dir_B / f'training_log_r{r}.csv', index=False)

        pred_model = MLPPLR(**arch_config).to(DEVICE)
        pred_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
        preds_B[r] = predict_proba_mlp_plr(pred_model, X_eval_t)
        per_rep_pr_auc_B[r] = float(average_precision_score(Y_eval, preds_B[r]))

        bundle = {
            'state_dict':  best_state,
            'arch_config': arch_config,
            'scaler':      num_scaler,
            'cat_scaler':  cat_scaler,
            'encoder':     encoder,
        }
        joblib.dump(bundle, dir_B / f'model_r{r}.joblib', compress=3)

        with open(dir_B / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)

        print(f'  B replica {r}: PR-AUC = {per_rep_pr_auc_B[r]:.4f}  '
              f'(best epoch {best_epoch}, val PR-AUC {best_val_pr:.4f})')

        del pred_model, X_tr_t, Y_tr_t, X_va_t, Y_va_t, X_eval_t
        torch.cuda.empty_cache()

    # ── Replica-averaged predictions and flagged set ──────────────────────
    p_hat_A = preds_A.mean(axis=0)
    p_hat_B = preds_B.mean(axis=0)
    flagged_local_idx = compute_flagged_topk(p_hat_A, p_hat_B, K_FRAC)

    pr_auc_A  = float(average_precision_score(Y_eval, p_hat_A))
    pr_auc_B  = float(average_precision_score(Y_eval, p_hat_B))
    roc_auc_A = float(roc_auc_score(Y_eval, p_hat_A))
    roc_auc_B = float(roc_auc_score(Y_eval, p_hat_B))

    print(f'  Averaged: PR-AUC A = {pr_auc_A:.4f}, PR-AUC B = {pr_auc_B:.4f}')
    print(f'            ROC-AUC A = {roc_auc_A:.4f}, ROC-AUC B = {roc_auc_B:.4f}')
    print(f'  Flagged (top {K_FRAC:.0%}): {flagged_local_idx.shape[0]:,} / {len(idx_eval):,}')

    # ── Save predictions NPZ (schema identical to nb 02) ─────────────────
    np.savez_compressed(
        pred_file,
        preds_A              = preds_A,
        preds_B              = preds_B,
        p_hat_A              = p_hat_A.astype(np.float32),
        p_hat_B              = p_hat_B.astype(np.float32),
        flagged_idx          = flagged_local_idx,
        Y_eval               = Y_eval,
        pr_auc_A             = np.float32(pr_auc_A),
        pr_auc_B             = np.float32(pr_auc_B),
        roc_auc_A            = np.float32(roc_auc_A),
        roc_auc_B            = np.float32(roc_auc_B),
        per_replica_pr_auc_A = per_rep_pr_auc_A,
        per_replica_pr_auc_B = per_rep_pr_auc_B,
    )

    with open(run_params_file, 'w') as f:
        json.dump(CURRENT_RUN_PARAMS, f, indent=2)

    performance_log.append({
        'pair_id':   pid,
        'pr_auc_A':  pr_auc_A,
        'pr_auc_B':  pr_auc_B,
        'roc_auc_A': roc_auc_A,
        'roc_auc_B': roc_auc_B,
        'n_flagged': flagged_local_idx.shape[0],
    })

print('\n✓ All window pairs processed.')

In [ ]:
perf_df = pd.DataFrame(performance_log)
print(perf_df.to_string(index=False))
print(f'\nMean PR-AUC  A: {perf_df["pr_auc_A"].mean():.4f}   B: {perf_df["pr_auc_B"].mean():.4f}')
print(f'Mean ROC-AUC A: {perf_df["roc_auc_A"].mean():.4f}   B: {perf_df["roc_auc_B"].mean():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_A'], 'o-',  label='Model A')
axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_B'], 's--', label='Model B')
axes[0].set_title('PR-AUC over window pairs')
axes[0].set_xlabel('Window pair'); axes[0].set_ylabel('PR-AUC')
axes[0].set_xticks(perf_df['pair_id'])
axes[0].legend(); axes[0].set_ylim(0, 1)

axes[1].plot(perf_df['pair_id'], perf_df['roc_auc_A'], 'o-',  label='Model A')
axes[1].plot(perf_df['pair_id'], perf_df['roc_auc_B'], 's--', label='Model B')
axes[1].set_title('ROC-AUC over window pairs')
axes[1].set_xlabel('Window pair'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_xticks(perf_df['pair_id'])
axes[1].legend(); axes[1].set_ylim(0.5, 1)

axes[2].bar(perf_df['pair_id'], perf_df['n_flagged'], color='tomato')
axes[2].set_title(f'Flagged instances per pair (top {int(K_FRAC*100)}%)')
axes[2].set_xlabel('Window pair'); axes[2].set_ylabel('|F_{A,B}|')
axes[2].set_xticks(perf_df['pair_id'])

plt.tight_layout()
plt.savefig(MODEL_DIR / 'performance_summary.png', dpi=120)
plt.show()

In [ ]:
perf_df.to_csv(MODEL_DIR / 'performance_log.csv', index=False)
print('Performance log saved.')